In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

file_path = '/content/drive/MyDrive/ROI_Features'

if os.path.exists(file_path):
    print("✅ File exists at:", file_path)
else:
    print("❌ File NOT found.")


❌ File NOT found.


In [ ]:
import os

folder_path = '/content/drive/My Drive/ROI_Features_Backup'

if os.path.exists(folder_path):
    print("✅ 'ROI_Features' folder exists.")
else:
    print("❌ 'ROI_Features' folder does not exist.")


✅ 'ROI_Features' folder exists.


In [ ]:
!unzip "/content/drive/MyDrive/Copy of hindi-visual-genome-11.zip" -d "/content/NMT"


Streaming output truncated to the last 5000 lines.
  inflating: /content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403839.jpg  
  inflating: /content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403843.jpg  
  inflating: /content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403846.jpg  
  inflating: /content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403849.jpg  
  inflating: /content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403852.jpg  
  inflating: /content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403856.jpg  
  inflating: /content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403860.jpg  
  inflating: /content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images/2403869.jpg

In [ ]:
import os

image_dir = "/content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images"

if os.path.exists(image_dir):
    print("✅ Directory exists:", image_dir)
else:
    print("❌ Directory NOT found:", image_dir)


✅ Directory exists: /content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images


In [ ]:
from PIL import Image
import os
import numpy as np
import torch
from torchvision import transforms, models
from tqdm import tqdm

# Directories
image_dir = "/content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.images"
output_feat_dir = "/content/NMT/dataset/ROI_Features"
os.makedirs(output_feat_dir, exist_ok=True)

# Image transform (match pretrained model input)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Use ResNet50 as feature extractor
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # remove final fc layer
resnet.eval().cuda()  # Use GPU if available


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 172MB/s]


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [ ]:
import pandas as pd

df = pd.read_csv('/content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.txt', sep='\t', header=None)
df.columns = ['image_id', 'x1', 'y1', 'w', 'h', 'en_caption', 'hi_caption']

feature_paths = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    img_path = os.path.join(image_dir, f"{row['image_id']}.jpg")
    save_path = os.path.join(output_feat_dir, f"{row['image_id']}_{i}.npy")

    try:
        img = Image.open(img_path).convert('RGB')

        # Crop and transform
        x1, y1, w, h = int(row['x1']), int(row['y1']), int(row['w']), int(row['h'])
        cropped = img.crop((x1, y1, x1 + w, y1 + h))
        tensor_img = transform(cropped).unsqueeze(0).cuda()

        # Extract features
        with torch.no_grad():
            feat = resnet(tensor_img).squeeze().cpu().numpy()  # shape: (2048,)
        np.save(save_path, feat)
        feature_paths.append(save_path)

    except Exception as e:
        print(f"⚠️ Skipping {row['image_id']} due to error: {e}")
        feature_paths.append(None)


 15%|█▌        | 4453/28930 [01:16<04:46, 85.46it/s]

⚠️ Skipping 2322111 due to error: invalid literal for int() with base 10: '29red sign on a pole'


 22%|██▏       | 6342/28930 [01:40<05:09, 73.08it/s]

⚠️ Skipping 2329621 due to error: invalid literal for int() with base 10: '168bike on beach'


 32%|███▏      | 9379/28930 [02:20<04:39, 70.01it/s]

⚠️ Skipping 2341369 due to error: invalid literal for int() with base 10: '367dog laying down'


 66%|██████▋   | 19179/28930 [04:26<01:57, 83.05it/s]

⚠️ Skipping 2379473 due to error: cannot convert float NaN to integer


100%|██████████| 28930/28930 [06:32<00:00, 73.64it/s]


In [ ]:
import os
import numpy as np

roi_dir = "/content/drive/MyDrive/ROI_Features_Backup"

# List all .npy files
all_files = [f for f in os.listdir(roi_dir) if f.endswith('.npy')]
total_files = len(all_files)

print(f"🧠 Total .npy ROI feature files: {total_files}")
print("📂 Sample files:", all_files[:5])

# Analyze file shapes
valid = 0
corrupt = 0
wrong_shape = 0
invalid_files = []

for file in all_files:
    path = os.path.join(roi_dir, file)
    try:
        arr = np.load(path)
        if arr.shape == (2048,):  # Expected ResNet50 feature shape
            valid += 1
        else:
            wrong_shape += 1
            invalid_files.append((file, arr.shape))
    except Exception as e:
        corrupt += 1
        invalid_files.append((file, str(e)))

print(f"\n✅ Valid feature files: {valid}")
print(f"❌ Corrupt files: {corrupt}")
print(f"⚠️ Files with wrong shape: {wrong_shape}")

# Show details of invalid ones
if invalid_files:
    print("\n🔍 Sample invalid entries:")
    for i, (name, detail) in enumerate(invalid_files[:5]):
        print(f"  {i+1}. {name} → {detail}")


🧠 Total .npy ROI feature files: 28926
📂 Sample files: ['2414002_27930.npy', '2414003_27931.npy', '2414004_27932.npy', '2414016_27933.npy', '2414019_27934.npy']


KeyboardInterrupt: 

In [ ]:
!cp -r /content/NMT/dataset/ROI_Features /content/drive/MyDrive/ROI_Features_Backup

In [ ]:
import os
import numpy as np

backup_dir = "/content/drive/MyDrive/ROI_Features_Backup"

# List all .npy files
all_files = [f for f in os.listdir(backup_dir) if f.endswith('.npy')]
total_files = len(all_files)

print(f"🧠 Total .npy feature files in backup: {total_files}")
print("📂 Sample files:", all_files[:5])

# Analyze file shapes
valid = 0
corrupt = 0
wrong_shape = 0
invalid_files = []

for file in all_files:
    path = os.path.join(backup_dir, file)
    try:
        arr = np.load(path)
        if arr.shape == (2048,):
            valid += 1
        else:
            wrong_shape += 1
            invalid_files.append((file, arr.shape))
    except Exception as e:
        corrupt += 1
        invalid_files.append((file, str(e)))

print(f"\n✅ Valid files: {valid}")
print(f"❌ Corrupt files: {corrupt}")
print(f"⚠️ Files with wrong shape: {wrong_shape}")

if invalid_files:
    print("\n🔍 Sample invalid entries:")
    for i, (fname, detail) in enumerate(invalid_files[:5]):
        print(f"  {i+1}. {fname} → {detail}")


🧠 Total .npy feature files in backup: 28926
📂 Sample files: ['11_0.npy', '17_1.npy', '18_2.npy', '21_3.npy', '23_4.npy']

✅ Valid files: 28926
❌ Corrupt files: 0
⚠️ Files with wrong shape: 0


In [ ]:
# Step 2: Define the path to the folder
import os

# Replace with the actual path inside your drive
backup_folder_path = "/content/drive/MyDrive/ROI_Features_Backup"

# Check if the path exists and list its contents
if os.path.exists(backup_folder_path):
    print("✅ Folder found. Contents:")
    print(os.listdir(backup_folder_path))
else:
    print("❌ Folder not found. Double-check the path.")


✅ Folder found. Contents:
['2414002_27930.npy', '2414003_27931.npy', '2414004_27932.npy', '2414016_27933.npy', '2414019_27934.npy', '2414021_27935.npy', '2414027_27936.npy', '2414029_27937.npy', '2414030_27938.npy', '2414048_27939.npy', '2414051_27940.npy', '2414055_27941.npy', '2414065_27942.npy', '2414069_27943.npy', '2414070_27944.npy', '2414073_27945.npy', '2414074_27946.npy', '2414083_27947.npy', '2414084_27948.npy', '2414086_27949.npy', '2414095_27950.npy', '2414098_27951.npy', '2414103_27952.npy', '2414104_27953.npy', '2414105_27954.npy', '2414106_27955.npy', '2414119_27956.npy', '2414121_27957.npy', '2414123_27958.npy', '2414127_27959.npy', '2414131_27960.npy', '2414137_27961.npy', '2414138_27962.npy', '2414147_27963.npy', '2414148_27964.npy', '2414159_27965.npy', '2414165_27966.npy', '2414179_27967.npy', '2414182_27968.npy', '2414184_27969.npy', '2414189_27970.npy', '2414190_27971.npy', '2414191_27972.npy', '2414200_27973.npy', '2414218_27974.npy', '2414223_27975.npy', '241422

In [ ]:
import os

# Path to your folder
folder_path = "/content/drive/MyDrive/ROI_Features_Backup"

# Count .npy files
npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
print(f"📦 Total .npy files: {len(npy_files)}")


📦 Total .npy files: 28926


In [ ]:
import numpy as np
import os

# Folder path
folder_path = "/content/drive/MyDrive/ROI_Features_Backup"

# List all .npy files
npy_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.npy')])

# Check if there is at least one .npy file
if not npy_files:
    print("❌ No .npy files found in the folder.")
else:
    # Pick the first one
    first_file = npy_files[0]
    file_path = os.path.join(folder_path, first_file)

    # Load the file
    data = np.load(file_path, allow_pickle=True)

    # Print structure
    print(f"📁 File name: {first_file}")
    print(f"🔹 Type: {type(data)}")

    try:
        print(f"🔹 Shape: {data.shape}")
    except AttributeError:
        print("🔹 No .shape attribute")

    # Print first item/content
    if hasattr(data, '__getitem__'):
        print("🔹 First item:", data[0])
    else:
        print("🔹 Content:", data)


📁 File name: 1003_257.npy
🔹 Type: <class 'numpy.ndarray'>
🔹 Shape: (2048,)
🔹 First item: 0.3987099


In [ ]:
import numpy as np
import os

folder_path = "/content/drive/MyDrive/ROI_Features_Backup"

# List .npy files
npy_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.npy')])

if not npy_files:
    print("❌ No .npy files found.")
else:
    file_name = npy_files[0]
    file_path = os.path.join(folder_path, file_name)

    # Load .npy file (could be ndarray, dict, or list)
    data = np.load(file_path, allow_pickle=True)

    print(f"\n📄 File: {file_name}")
    print(f"🔹 Type: {type(data)}")

    try:
        print(f"🔹 Shape: {data.shape}")
    except AttributeError:
        print("🔹 No shape attribute (likely a dict, list, or object array)")

    # Display first item or keys
    if isinstance(data, dict):
        print("🔹 Keys:", list(data.keys()))
    elif isinstance(data, (list, np.ndarray)) and len(data) > 0:
        print("🔹 First item type:", type(data[0]))
        print("🔹 First item (truncated):", str(data[0])[:300])
    else:
        print("🔹 Content:", data)



📄 File: 1003_257.npy
🔹 Type: <class 'numpy.ndarray'>
🔹 Shape: (2048,)
🔹 First item type: <class 'numpy.float32'>
🔹 First item (truncated): 0.3987099


In [ ]:
import json

# Input text file path (replace with actual if needed)
input_txt_path = "/content/NMT/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.txt"
output_json_path = "/content/image_ids_and_captions.json"

output_list = []

with open(input_txt_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")

        if len(parts) >= 7:
            image_id = parts[0].strip()
            english_caption = parts[-2].strip()
            hindi_caption = parts[-1].strip()

            output_list.append({
                "image_id": image_id,
                "english_caption": english_caption,
                "hindi_caption": hindi_caption
            })

# Write to JSON
with open(output_json_path, "w", encoding="utf-8") as f_out:
    json.dump(output_list, f_out, ensure_ascii=False, indent=2)

print(f"✅ JSON saved to: {output_json_path}")


✅ JSON saved to: /content/image_ids_and_captions.json


In [ ]:
import os
import torch
from torch.utils.data import Dataset
import sentencepiece as spm
import numpy as np
import json

class LazyImageCaptionDataset(Dataset):
    def __init__(self, json_path, npy_dir, sp_model_path, max_len=30, pad_token=0):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        self.npy_dir = npy_dir
        self.max_len = max_len
        self.pad_token = pad_token

        self.sp = spm.SentencePieceProcessor()
        self.sp.load(sp_model_path)
        self.bos = self.sp.bos_id()
        self.eos = self.sp.eos_id()

        # ✅ Pre-index all .npy file paths by image_id
        self.imageid_to_path = {}
        for f in os.listdir(npy_dir):
            if f.endswith(".npy"):
                image_id = f.split("_")[0]
                self.imageid_to_path[image_id] = os.path.join(npy_dir, f)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]
        image_id = entry["image_id"]
        english_caption = entry["english_caption"].strip()

        # ✅ Fast lookup
        npy_path = self.imageid_to_path.get(image_id)
        if npy_path is None:
            raise FileNotFoundError(f"No .npy file found for image_id {image_id}")

        feature = np.load(npy_path)
        if len(feature.shape) == 2:
            feature = np.mean(feature, axis=0)

        tokens = self.sp.encode(english_caption, out_type=int)
        decoder_input = [self.bos] + tokens
        decoder_target = tokens + [self.eos]

        decoder_input = decoder_input[:self.max_len] + [self.pad_token] * (self.max_len - len(decoder_input))
        decoder_target = decoder_target[:self.max_len] + [self.pad_token] * (self.max_len - len(decoder_target))

        return {
            'image': torch.tensor(feature, dtype=torch.float32),
            'decoder_input': torch.tensor(decoder_input, dtype=torch.long),
            'decoder_target': torch.tensor(decoder_target, dtype=torch.long)
        }


In [ ]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=3, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers)
        self.image_proj = nn.Linear(2048, d_model)  # ✅ ResNet output size
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, memory, tgt_mask, tgt_padding_mask):
        tgt_emb = self.embedding(tgt)
        tgt_emb = self.pos_encoder(tgt_emb)
        output = self.decoder(tgt=tgt_emb,
                              memory=memory,
                              tgt_mask=tgt_mask,
                              tgt_key_padding_mask=tgt_padding_mask)
        return self.fc_out(output)


In [ ]:
def generate_square_subsequent_mask(sz):
    mask = (torch.triu(torch.ones((sz, sz))) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask


In [ ]:
def train_model(model, dataloader, optimizer, criterion, pad_token, num_epochs=10):
    import torch.nn.utils as utils
    from tqdm import tqdm

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        loop = tqdm(dataloader, desc=f"Epoch {epoch+1}")

        for batch in loop:
            img = batch['image'].to(device)
            dec_in = batch['decoder_input'].to(device)
            dec_out = batch['decoder_target'].to(device)

            tgt = dec_in.transpose(0, 1)  # [T, B]
            tgt_y = dec_out.transpose(0, 1)
            memory = model.image_proj(img).unsqueeze(0)  # [1, B, d_model]

            tgt_mask = generate_square_subsequent_mask(tgt.size(0)).to(device)
            tgt_padding_mask = (dec_in == pad_token).to(device)

            logits = model(tgt, memory, tgt_mask, tgt_padding_mask)
            logits = logits.view(-1, logits.size(-1))
            tgt_y = tgt_y.reshape(-1)

            loss = criterion(logits, tgt_y)
            optimizer.zero_grad()
            loss.backward()
            utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            loop.set_postfix(loss=total_loss / (loop.n + 1e-8))

        print(f"✅ Epoch {epoch+1} | Avg Loss: {total_loss / len(dataloader):.4f}")


In [ ]:
from torch.utils.data import DataLoader
import torch.nn as nn
import torch

# Paths
json_path = "/content/image_ids_and_captions.json"
npy_dir = "/content/drive/MyDrive/ROI_Features_Backup"
sp_model_path = "/content/english_unigram_4500.model"

# Dataset + Loader
train_dataset = LazyImageCaptionDataset(json_path, npy_dir, sp_model_path, pad_token=0)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Model
model = TransformerDecoder(vocab_size=4500).to("cuda")
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

# Train
train_model(model, train_loader, optimizer, criterion, pad_token=0, num_epochs=30)


Epoch 1:   0%|          | 0/905 [04:14<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import time

start = time.time()
_ = train_dataset[0]
print(f"⏱️ First sample loaded in {time.time() - start:.2f} sec")


⏱️ First sample loaded in 4.10 sec


In [ ]:
import os
import numpy as np
import pickle
from tqdm import tqdm

# 📂 Input .npy directory and output .pkl file path
npy_dir = "/content/drive/MyDrive/ROI_Features_Backup"
output_pkl = "/content/resnet_features.pkl"

# 📦 Container for all features
feature_dict = {}

# 📜 Get list of all .npy files
npy_files = sorted([f for f in os.listdir(npy_dir) if f.endswith(".npy")])

# 🔄 Loop and load
for fname in tqdm(npy_files, desc="🔄 Converting .npy to .pkl"):
    image_id = fname.split("_")[0]  # e.g., 2373931 from 2373931_17733.npy
    npy_path = os.path.join(npy_dir, fname)

    try:
        feature = np.load(npy_path)

        # 📏 If region-level (N, 2048), pool to (2048,)
        if len(feature.shape) == 2:
            feature = np.mean(feature, axis=0)

        feature_dict[image_id] = feature

    except Exception as e:
        print(f"⚠️ Error loading {fname}: {e}")

# 💾 Save the full feature dictionary as .pkl
with open(output_pkl, "wb") as f:
    pickle.dump(feature_dict, f)

print(f"\n✅ Saved {len(feature_dict)} image features to: {output_pkl}")


🔄 Converting .npy to .pkl: 100%|██████████| 28926/28926 [01:24<00:00, 340.32it/s]



✅ Saved 28923 image features to: /content/resnet_features.pkl


In [ ]:
import shutil

# Source: Colab local path
src = "/content/resnet_features.pkl"

# Destination: Your Google Drive
dst = "/content/drive/MyDrive/resnet_features.pkl"

# Copy the file
shutil.copy(src, dst)

print(f"✅ File moved to Google Drive: {dst}")


✅ File moved to Google Drive: /content/drive/MyDrive/resnet_features.pkl


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ CUDA is available: {torch.cuda.get_device_name(0)}")
else:
    print("❌ CUDA is not available. Using CPU.")


✅ CUDA is available: Tesla T4


In [ ]:
import pickle
import numpy as np

# Load the pickle file
pkl_path = "/content/drive/MyDrive/resnet_features.pkl"

with open(pkl_path, "rb") as f:
    features = pickle.load(f)

# Check total entries
print(f"✅ Total image features: {len(features)}")

# Inspect structure of the first few entries
for i, (image_id, feature) in enumerate(features.items()):
    print(f"\n🔎 Entry {i+1}")
    print(f"Image ID: {image_id}")
    print(f"Feature shape: {feature.shape}")
    print(f"Feature dtype: {feature.dtype}")
    print(f"Feature sample: {feature[:5]}")  # Print first 5 values

    if i >= 2:  # Show only first 3 entries
        break


✅ Total image features: 28923

🔎 Entry 1
Image ID: 1003
Feature shape: (2048,)
Feature dtype: float32
Feature sample: [0.3987099  0.46104223 0.01151201 0.00686998 0.14986204]

🔎 Entry 2
Image ID: 1004
Feature shape: (2048,)
Feature dtype: float32
Feature sample: [0.2579017  0.5832698  0.72753876 0.15664008 1.840999  ]

🔎 Entry 3
Image ID: 1005
Feature shape: (2048,)
Feature dtype: float32
Feature sample: [0.34698564 0.22215901 0.05418744 0.15107156 0.8593441 ]


In [ ]:
import torch
from torch.utils.data import Dataset
import pickle
import sentencepiece as spm
import json

class PklFeatureCaptionDataset(Dataset):
    def __init__(self, json_path, pkl_feature_path, sp_model_path, max_len=30, pad_token=0):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        with open(pkl_feature_path, "rb") as f:
            self.image_features = pickle.load(f)

        self.sp = spm.SentencePieceProcessor()
        self.sp.load(sp_model_path)
        self.bos = self.sp.bos_id()
        self.eos = self.sp.eos_id()

        self.max_len = max_len
        self.pad_token = pad_token

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]
        image_id = entry["image_id"]
        caption = entry["english_caption"].strip()

        if image_id not in self.image_features:
            raise KeyError(f"Missing image feature for ID: {image_id}")

        feature = self.image_features[image_id]

        tokens = self.sp.encode(caption, out_type=int)
        decoder_input = [self.bos] + tokens
        decoder_target = tokens + [self.eos]

        decoder_input = decoder_input[:self.max_len] + [self.pad_token] * (self.max_len - len(decoder_input))
        decoder_target = decoder_target[:self.max_len] + [self.pad_token] * (self.max_len - len(decoder_target))

        return {
            'image': torch.tensor(feature, dtype=torch.float32),
            'decoder_input': torch.tensor(decoder_input, dtype=torch.long),
            'decoder_target': torch.tensor(decoder_target, dtype=torch.long)
        }


In [ ]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=3, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers)
        self.image_proj = nn.Linear(2048, d_model)  # ResNet features
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, memory, tgt_mask, tgt_padding_mask):
        tgt_emb = self.embedding(tgt)
        tgt_emb = self.pos_encoder(tgt_emb)
        output = self.decoder(tgt=tgt_emb,
                              memory=memory,
                              tgt_mask=tgt_mask,
                              tgt_key_padding_mask=tgt_padding_mask)
        return self.fc_out(output)


In [ ]:
from tqdm import tqdm

def generate_square_subsequent_mask(sz):
    mask = torch.triu(torch.ones((sz, sz)) == 1).transpose(0, 1)
    return mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, 0.0)


def train_model(model, dataloader, optimizer, criterion, pad_token, num_epochs=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        loop = tqdm(dataloader, desc=f"🧠 Epoch {epoch+1}/{num_epochs}")

        for batch in loop:
            img = batch['image'].to(device)
            dec_in = batch['decoder_input'].to(device)
            dec_out = batch['decoder_target'].to(device)

            tgt = dec_in.transpose(0, 1)
            tgt_y = dec_out.transpose(0, 1)
            memory = model.image_proj(img).unsqueeze(0)

            tgt_mask = generate_square_subsequent_mask(tgt.size(0)).to(device)
            tgt_padding_mask = (dec_in == pad_token)

            logits = model(tgt, memory, tgt_mask, tgt_padding_mask)
            logits = logits.view(-1, logits.size(-1))
            tgt_y = tgt_y.reshape(-1)

            loss = criterion(logits, tgt_y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            loop.set_postfix({
                "Batch Loss": loss.item(),
                "Avg Loss": total_loss / (loop.n + 1e-8)
            })

        print(f"\n✅ Epoch {epoch+1} completed. Average Loss: {total_loss / len(dataloader):.4f}\n")


In [ ]:
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

# Paths
json_path = "/content/image_ids_and_captions.json"
pkl_feature_path = "/content/drive/MyDrive/resnet_features.pkl"
sp_model_path = "/content/english_unigram_4500.model"

# Load dataset
dataset = PklFeatureCaptionDataset(json_path, pkl_feature_path, sp_model_path, max_len=30, pad_token=0)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Load model
vocab_size = 4500  # Update if yours is different
model = TransformerDecoder(vocab_size=vocab_size)

# Train
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
train_model(model, train_loader, optimizer, criterion, pad_token=0, num_epochs=10)


NameError: name 'PklFeatureCaptionDataset' is not defined

In [ ]:
import torch
from torch.utils.data import Dataset
import pickle
import sentencepiece as spm
import json

class PklFeatureCaptionDataset(Dataset):
    def __init__(self, json_path, pkl_feature_path, sp_model_path, max_len=30, pad_token=0):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        with open(pkl_feature_path, "rb") as f:
            self.image_features = pickle.load(f)

        self.sp = spm.SentencePieceProcessor()
        self.sp.load(sp_model_path)
        self.bos = self.sp.bos_id()
        self.eos = self.sp.eos_id()
        self.vocab_size = self.sp.get_piece_size()

        self.max_len = max_len
        self.pad_token = pad_token

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]
        image_id = entry["image_id"]
        caption = entry["english_caption"].strip()

        if image_id not in self.image_features:
            raise KeyError(f"Missing feature for image ID: {image_id}")

        feature = self.image_features[image_id]
        tokens = self.sp.encode(caption, out_type=int)

        decoder_input = [self.bos] + tokens
        decoder_target = tokens + [self.eos]

        decoder_input = decoder_input[:self.max_len] + [self.pad_token] * (self.max_len - len(decoder_input))
        decoder_target = decoder_target[:self.max_len] + [self.pad_token] * (self.max_len - len(decoder_target))

        return {
            'image': torch.tensor(feature, dtype=torch.float32),
            'decoder_input': torch.tensor(decoder_input, dtype=torch.long),
            'decoder_target': torch.tensor(decoder_target, dtype=torch.long)
        }


In [ ]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=3, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers)
        self.image_proj = nn.Linear(2048, d_model)  # ResNet features
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, memory, tgt_mask, tgt_padding_mask):
        tgt_emb = self.embedding(tgt)
        tgt_emb = self.pos_encoder(tgt_emb)
        output = self.decoder(tgt=tgt_emb,
                              memory=memory,
                              tgt_mask=tgt_mask,
                              tgt_key_padding_mask=tgt_padding_mask)
        return self.fc_out(output)


In [ ]:
from tqdm import tqdm
import torch.nn.functional as F

def generate_square_subsequent_mask(sz):
    mask = torch.triu(torch.ones((sz, sz)) == 1).transpose(0, 1)
    return mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, 0.0)

def train_model(model, dataloader, optimizer, criterion, pad_token, num_epochs=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0.0
        progress = tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=True)

        for batch in progress:
            img = batch['image'].to(device)                    # [B, 2048]
            dec_in = batch['decoder_input'].to(device)         # [B, T]
            dec_out = batch['decoder_target'].to(device)       # [B, T]

            tgt = dec_in.transpose(0, 1)                        # [T, B]
            tgt_y = dec_out.transpose(0, 1)                     # [T, B]

            memory = model.image_proj(img).unsqueeze(0)        # [1, B, d_model]

            tgt_mask = generate_square_subsequent_mask(tgt.size(0)).to(device)
            tgt_padding_mask = (dec_in == pad_token)           # [B, T] → OK

            # Forward pass
            logits = model(tgt, memory, tgt_mask, tgt_padding_mask)
            logits = logits.view(-1, logits.size(-1))          # [T*B, vocab]
            tgt_y = tgt_y.reshape(-1)                          # [T*B]

            # Guard for invalid targets
            if tgt_y.max() >= logits.size(-1):
                print(f"❌ Token ID {tgt_y.max().item()} out of range.")
                continue

            loss = criterion(logits, tgt_y)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix(loss=loss.item())

        print(f"✅ Epoch {epoch+1} finished | Avg Loss: {total_loss / len(dataloader):.4f}")



In [ ]:
max_seen = 0
bad_entries = []

for i in range(len(dataset)):
    entry = dataset[i]
    dec_in_max = entry['decoder_input'].max().item()
    dec_tgt_max = entry['decoder_target'].max().item()
    if dec_in_max >= 4500 or dec_tgt_max >= 4500:
        bad_entries.append((i, dec_in_max, dec_tgt_max))
    max_seen = max(max_seen, dec_in_max, dec_tgt_max)

print(f"✅ Maximum token ID in dataset: {max_seen}")
print(f"⚠️ Problematic entries: {bad_entries[:5]} (showing only first 5)")
print(f"🔢 Total bad entries: {len(bad_entries)}")


✅ Maximum token ID in dataset: 4490
⚠️ Problematic entries: [] (showing only first 5)
🔢 Total bad entries: 0


In [ ]:
import json

# Paths
original_json_path = "/content/image_ids_and_captions.json"
cleaned_json_path = "/content/image_ids_and_captions_filtered.json"

# Image IDs to remove (as strings)
remove_ids = {"2322111", "2329621", "2341369", "2379473"}

# Load JSON
with open(original_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Filter entries
filtered_data = [entry for entry in data if entry["image_id"] not in remove_ids]

# Save filtered JSON
with open(cleaned_json_path, "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print(f"✅ Removed {len(data) - len(filtered_data)} entries.")
print(f"✅ Filtered JSON saved to: {cleaned_json_path}")


✅ Removed 4 entries.
✅ Filtered JSON saved to: /content/image_ids_and_captions_filtered.json


In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
import sentencepiece as spm

# File paths
json_path = "/content/image_ids_and_captions_filtered.json"
pkl_path = "/content/drive/MyDrive/resnet_features.pkl"
spm_path = "/content/english_unigram_4500.model"

# Load tokenizer and get vocab size
vocab_size = max(
    sp.get_piece_size(),
    sp.bos_id() + 1,
    sp.eos_id() + 1,
    sp.pad_id() + 1 if sp.pad_id() != -1 else 0
)
print("✅ Safe vocab size:", vocab_size)

for i in range(5):  # Check first few batches
    batch = next(iter(train_loader))

    img = batch['image']
    dec_in = batch['decoder_input']
    dec_out = batch['decoder_target']

    print(f"✅ Batch {i}")
    print("Image shape:", img.shape)
    print("Decoder input max token:", dec_in.max().item())
    print("Decoder target max token:", dec_out.max().item())



# Load dataset + dataloader
dataset = PklFeatureCaptionDataset(json_path, pkl_path, spm_path, max_len=30, pad_token=0)
train_loader = DataLoader(dataset, batch_size=1, shuffle=True)

# Init model
model = TransformerDecoder(vocab_size=vocab_size)

# Setup training
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

# Train
train_model(model, train_loader, optimizer, criterion, pad_token=0, num_epochs=10)


✅ Safe vocab size: 4500
✅ Batch 0
Image shape: torch.Size([1, 2048])
Decoder input max token: 284
Decoder target max token: 284
✅ Batch 1
Image shape: torch.Size([1, 2048])
Decoder input max token: 178
Decoder target max token: 178
✅ Batch 2
Image shape: torch.Size([1, 2048])
Decoder input max token: 361
Decoder target max token: 361
✅ Batch 3
Image shape: torch.Size([1, 2048])
Decoder input max token: 530
Decoder target max token: 530
✅ Batch 4
Image shape: torch.Size([1, 2048])
Decoder input max token: 671
Decoder target max token: 671


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
